In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:
# Build a .py script that takes a snapshot date, trains a model and outputs artefact into storage.

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/17 11:33:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/17 11:33:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## set up config

In [4]:
model_train_date_str = os.getenv("MODEL_TRAIN_DATE", "2024-09-01")
train_test_period_months = int(os.getenv("TRAIN_TEST_MONTHS", 12))
oot_period_months       = int(os.getenv("OOT_MONTHS", 2))
train_test_ratio        = float(os.getenv("TRAIN_TEST_RATIO", 0.8))

config = {}
config["model_train_date_str"]    = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"]        = oot_period_months

config["model_train_date"] = datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"]     = config["model_train_date"] - timedelta(days=1)
config["oot_start_date"]   = config["model_train_date"] - relativedelta(months=oot_period_months)
config["train_test_end_date"]   = config["oot_start_date"] - timedelta(days=1)
config["train_test_start_date"] = config["oot_start_date"] - relativedelta(months=train_test_period_months)
config["train_test_ratio"]       = train_test_ratio

# helper to format back to ISO strings for Spark filters
def fmt(d): 
    return d.strftime("%Y-%m-%d")

# derive all split dates
train_start = config["train_test_start_date"]
train_end   = config["train_test_end_date"]
val_date    = train_end  + relativedelta(months=1)
test_date   = val_date   + relativedelta(months=1)
oot1_date   = config["oot_start_date"]
oot2_date   = config["oot_end_date"]
prod_nov    = oot2_date  + relativedelta(months=1)
prod_dec    = prod_nov   + relativedelta(months=1)

## get label store

In [5]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [6]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

# your “cutoff” for model training
mtd = datetime.strptime(config["model_train_date_str"], "%Y-%m-%d")

# TRAIN/VAL/TEST window is the 12 months immediately preceding the 2-month OOT window
# so your TRAIN window starts 14 months before mtd
train_start = mtd - relativedelta(months=config["oot_period_months"] + config["train_test_period_months"])
# TRAIN runs 12 months from there
train_end   = train_start + relativedelta(months=config["train_test_period_months"] - 1)

# the next three single‐month dates
val_date  = train_end + relativedelta(months=1)
test_date = val_date    + relativedelta(months=1)
oot1_date = test_date   + relativedelta(months=1)
oot2_date = oot1_date   + relativedelta(months=1)

# two production months
prod_nov_date = oot2_date + relativedelta(months=1)
prod_dec_date = prod_nov_date + relativedelta(months=1)

# helper to format for Spark filters
def fmt(d): return d.strftime("%Y-%m-%d")

from pyspark.sql.functions import col

labels = {
    "TRAIN": label_store_sdf.filter(
        (col("snapshot_date") >= fmt(train_start)) &
        (col("snapshot_date") <= fmt(train_end))
    ),
    "VAL":   label_store_sdf.filter(col("snapshot_date") == fmt(val_date)),
    "TEST":  label_store_sdf.filter(col("snapshot_date") == fmt(test_date)),
    "OOT1":  label_store_sdf.filter(col("snapshot_date") == fmt(oot1_date)),
    "OOT2":  label_store_sdf.filter(col("snapshot_date") == fmt(oot2_date)),
    "PROD_NOV": label_store_sdf.filter(col("snapshot_date") == fmt(prod_nov_date)),
    "PROD_DEC": label_store_sdf.filter(col("snapshot_date") == fmt(prod_dec_date)),
}

# sanity check
for name, sdf in labels.items():
    print(f"{name} labels:", sdf.count())

TRAIN labels: 5958
VAL labels: 485
TEST labels: 518
OOT1 labels: 511
OOT2 labels: 513
PROD_NOV labels: 491
PROD_DEC labels: 498


## get features

In [7]:
feature_location = "data/gold/feature/"

feature_path = "datamart/gold/feature/*.parquet"

features_store_sdf = (
    spark.read
         .option("mergeSchema", "true")   # optional, if you have evolving schemas
         .parquet(feature_path)
)

print("row_count:", features_store_sdf.count())
features_store_sdf.show(truncate=False)


row_count: 218902
+-----------+-------------+------+-----+-----+-----+-----+-----+------+------+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+------+------+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+-----------------+---------------+----------------------+--------------------+---------------------------+-----------------------------+-------------------------+---------------+-------------------------+-----------------------------+----------------------+-------------------+-------------------+-----------------+-------------------+------------------+-------------+----+----+----------+-------+-----------+
|Customer_ID|snapshot_date|fe_1  |fe_2 |fe_3 |fe_4 |fe_5 |fe_6 |fe_7  |fe_8  |fe_9 |fe_10|fe_1

In [8]:
feature_splits = {
    "TRAIN":   ((train_start, train_end)),
    "VAL":     ((val_date,    val_date)),
    "TEST":    ((test_date,   test_date)),
    "OOT1":    ((oot1_date,   oot1_date)),
    "OOT2":    ((oot2_date,   oot2_date)),
    "PROD_NOV":((prod_nov_date, prod_nov_date)),
    "PROD_DEC":((prod_dec_date, prod_dec_date)),
}

df_features = {}
for name, (start_dt, end_dt) in feature_splits.items():
    df_features[name] = features_store_sdf.filter(
        (col("snapshot_date") >= fmt(start_dt)) &
        (col("snapshot_date") <= fmt(end_dt))
    )
    print(f"{name} features:", df_features[name].count())

TRAIN features: 107688
VAL features: 9479
TEST features: 9517
OOT1 features: 9467
OOT2 features: 9430
PROD_NOV features: 9462
PROD_DEC features: 9489


## prepare data for modeling

In [9]:
from pyspark.sql import SparkSession, functions as F
from functools import reduce

spark = (
    SparkSession.builder
    .appName("JoinFeaturesLabels")
    .getOrCreate()
)

label_path = "datamart/gold/label_store/*.parquet"
labels_sdf = (
    spark.read
         .option("mergeSchema", "true")
         .parquet(label_path)
         .withColumn("snapshot_date", F.to_date("snapshot_date"))
)

feature_path = "datamart/gold/feature/*.parquet"
features_sdf = (
    spark.read
         .option("mergeSchema", "true")
         .parquet(feature_path)
         .withColumn("snapshot_date", F.to_date("snapshot_date"))
)

print("labels_sdf rows:  ", labels_sdf.count())
print("raw features_sdf rows:", features_sdf.count())

feature_cols = [c for c in features_sdf.columns if c.startswith("fe_")]

features_trimmed = features_sdf.select(
    "Customer_ID", "snapshot_date", *feature_cols
)

print("trimmed features_sdf rows:", features_trimmed.count(), "columns:", feature_cols)

# Left-join features onto labels
joined_sdf = labels_sdf.join(
    features_trimmed,
    on=["Customer_ID", "snapshot_date"],
    how="left"
)
print("after join:", joined_sdf.count())

nonnull_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNotNull() for c in feature_cols]
)

final_sdf = joined_sdf.filter(nonnull_condition)
print("after dropping all-null features:", final_sdf.count())

final_sdf.write.mode("overwrite").parquet("/data/final_merged/")
print("Wrote final merged table to /data/final_merged/")

labels_sdf rows:   8974
raw features_sdf rows: 218902
trimmed features_sdf rows: 218902 columns: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'fe_1_5_mean']
after join: 8974
after dropping all-null features: 8974
Wrote final merged table to /data/final_merged/


In [16]:
import pandas as pd

file_path = "data/final_merged.csv"
df = pd.read_csv(file_path, parse_dates=['snapshot_date'], low_memory=False)

print(f"Loaded '{file_path}' with shape: {df.shape}")
display(df.head())

Loaded 'data/final_merged.csv' with shape: (8974, 64)


,loan_id,Customer_ID,label,label_def,snapshot_date,fe_1,fe_2,fe_3,fe_4,fe_5,...,Auto_Loan_count,Credit_Builder_Loan_count,Debt_Consolidation_Loan_count,Home_Equity_Loan_count,Mortgage_Loan_count,Not_Specified_count,Payday_Loan_count,Personal_Loan_count,Student_Loan_count,Unknown_count
0,CUS_0x1037_2023_01_01,CUS_0x1037,0,30dpd_6mob,2023-07-01,40.0,90.0,231.0,136.0,239.0,...,2.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,CUS_0x1069_2023_01_01,CUS_0x1069,0,30dpd_6mob,2023-07-01,-26.0,37.0,87.0,26.0,173.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,CUS_0x114a_2023_01_01,CUS_0x114a,0,30dpd_6mob,2023-07-01,35.0,230.0,102.0,146.0,-40.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,CUS_0x1184_2023_01_01,CUS_0x1184,0,30dpd_6mob,2023-07-01,278.0,-50.0,-132.0,209.0,-52.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,CUS_0x1297_2023_01_01,CUS_0x1297,1,30dpd_6mob,2023-07-01,162.0,190.0,327.0,96.0,239.0,...,0.0,1.0,0.0,1.0,1.0,0.0,3.0,2.0,1.0,0.0


In [17]:
# Define the key snapshot dates
train_start = pd.Timestamp(2023, 7, 1)
train_end   = train_start + pd.DateOffset(months=11)   # → 2024-06-01

val_date  = train_end  + pd.DateOffset(months=1)       # → 2024-07-01
test_date = val_date    + pd.DateOffset(months=1)       # → 2024-08-01
oot1_date = test_date   + pd.DateOffset(months=1)       # → 2024-09-01
oot2_date = oot1_date   + pd.DateOffset(months=1)       # → 2024-10-01

sim_nov_date = oot2_date   + pd.DateOffset(months=1)    # → 2024-11-01
sim_dec_date = sim_nov_date + pd.DateOffset(months=1)   # → 2024-12-01

train = df[(df.snapshot_date >= train_start) & (df.snapshot_date <= train_end)]
val   = df[df.snapshot_date == val_date]
test  = df[df.snapshot_date == test_date]
oot1  = df[df.snapshot_date == oot1_date]
oot2  = df[df.snapshot_date == oot2_date]
prod_nov = df[df.snapshot_date == sim_nov_date]
prod_dec = df[df.snapshot_date == sim_dec_date]

# Features & labels
drop_cols = ['Customer_ID','snapshot_date','label_def','loan_id','Name','SSN']
features  = [c for c in train.columns if c not in drop_cols + ['label']]

X_train, y_train = train[features], train['label']
X_val,   y_val   = val  [features], val  ['label']
X_test,  y_test  = test [features], test ['label']
X_oot1,  y_oot1  = oot1 [features], oot1 ['label']
X_oot2,  y_oot2  = oot2 [features], oot2 ['label']
X_nov,   y_nov   = prod_nov[features], prod_nov['label']
X_dec,   y_dec   = prod_dec[features], prod_dec['label']

splits = [
    ("TRAIN",     X_train, y_train),
    ("VAL",       X_val,   y_val),
    ("TEST",      X_test,  y_test),
    ("OOT1",      X_oot1,  y_oot1),
    ("OOT2",      X_oot2,  y_oot2),
    ("PROD_NOV",  X_nov,   y_nov),
    ("PROD_DEC",  X_dec,   y_dec),
]

In [18]:
import os, random
import numpy as np
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [20]:
## model best parameters were obtained previously and now refit again for training

import numpy as np
import xgboost as xgb

from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline          import Pipeline
from sklearn.metrics           import (
    roc_auc_score,
    fbeta_score,
    classification_report
)

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
preproc = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="drop")

best_params = {
    "n_estimators":     415,
    "max_depth":        3,
    "learning_rate":    0.009057293740640768,
    "subsample":        0.7276734855131275,
    "colsample_bytree": 0.6232189363962957,
    "reg_alpha":        0.9502195804742386,
    "reg_lambda":       1.539615751287922,
    "eval_metric":      "logloss",
    "random_state":     42,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
}
best_k = 19  
beta   = 2.0


pipe_final = Pipeline([
    ("pre",    preproc),
    ("select", SelectKBest(mutual_info_classif, k=best_k)),
    ("clf",    xgb.XGBClassifier(**best_params))
])
pipe_final.fit(X_train, y_train)


best_thresh = 0.32
print(f"Using fixed threshold for F{beta:.0f}: {best_thresh:.2f}")

def evaluate(name, X, y):
    proba = pipe_final.predict_proba(X)[:, 1]
    preds = (proba >= best_thresh).astype(int)
    auc   = roc_auc_score(y, proba)
    gini  = 2 * auc - 1
    f2    = fbeta_score(y, preds, beta=beta)
    print(f"\n{name} @ thresh={best_thresh:.2f}")
    print(f"  F{beta:.0f}: {f2:.4f}  AUC: {auc:.4f}  Gini: {gini:.4f}")
    print(classification_report(y, preds, digits=4))

for split_name, (X, y) in [
    ("TRAIN", (X_train, y_train)),
    ("VAL",   (X_val,   y_val)),
    ("TEST",  (X_test,  y_test)),
    ("OOT1",  (X_oot1,  y_oot1)),
    ("OOT2",  (X_oot2,  y_oot2)),
]:
    evaluate(split_name, X, y)

feat_names = pipe_final.named_steps["pre"].get_feature_names_out()
mask       = pipe_final.named_steps["select"].get_support()
kept       = feat_names[mask]
dropped    = feat_names[~mask]

print(f"\nKept {len(kept)} features:", kept.tolist())
print(f"Dropped {len(dropped)} features:", dropped.tolist())

Using fixed threshold for F2: 0.32

TRAIN @ thresh=0.32
  F2: 0.7267  AUC: 0.8247  Gini: 0.6493
              precision    recall  f1-score   support

           0     0.9106    0.6103    0.7308      4272
           1     0.4620    0.8482    0.5982      1686

    accuracy                         0.6776      5958
   macro avg     0.6863    0.7292    0.6645      5958
weighted avg     0.7837    0.6776    0.6933      5958


VAL @ thresh=0.32
  F2: 0.7186  AUC: 0.8006  Gini: 0.6013
              precision    recall  f1-score   support

           0     0.8917    0.6276    0.7367       341
           1     0.4816    0.8194    0.6067       144

    accuracy                         0.6845       485
   macro avg     0.6866    0.7235    0.6717       485
weighted avg     0.7699    0.6845    0.6981       485


TEST @ thresh=0.32
  F2: 0.6831  AUC: 0.7870  Gini: 0.5741
              precision    recall  f1-score   support

           0     0.8809    0.5580    0.6832       371
           1     0.420

In [21]:
# import optuna
# import numpy as np
# import xgboost as xgb
# import pandas as pd

# from sklearn.compose           import ColumnTransformer
# from sklearn.preprocessing     import StandardScaler, OneHotEncoder
# from sklearn.feature_selection import SelectKBest, mutual_info_classif
# from sklearn.pipeline          import Pipeline
# from sklearn.model_selection   import StratifiedKFold, cross_val_score
# from sklearn.metrics           import (
#     roc_auc_score,
#     fbeta_score,
#     classification_report,
#     make_scorer,
# )

# # 1) Preprocessor
# num_cols = X_train.select_dtypes(include="number").columns.tolist()
# cat_cols = X_train.select_dtypes(include="object").columns.tolist()
# preproc   = ColumnTransformer([
#     ("num", StandardScaler(), num_cols),
#     ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
# ], remainder="drop")

# # 2) Count post-preproc features
# X_tr_t     = preproc.fit_transform(X_train)
# N_FEATURES = X_tr_t.shape[1]

# beta = 2
# f_beta_scorer = make_scorer(fbeta_score, beta=beta)

# # 4) Optuna objective
# def objective(trial):
#     k = trial.suggest_int("k", int(0.2*N_FEATURES), N_FEATURES)
#     params = {
#         "n_estimators":     trial.suggest_int("n_estimators", 50, 500),
#         "max_depth":        trial.suggest_int("max_depth", 3, 10),
#         "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
#         "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 1.0),
#         "reg_lambda":       trial.suggest_float("reg_lambda", 1.0, 3.0),
#         "eval_metric":      "logloss",
#         "random_state":     42,
#         "scale_pos_weight": (y_train==0).sum()/(y_train==1).sum(),
#     }
#     pipe = Pipeline([
#         ("pre",    preproc),
#         ("select", SelectKBest(mutual_info_classif, k=k)),
#         ("clf",    xgb.XGBClassifier(**params))
#     ])
#     cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     scores = cross_val_score(pipe, X_train, y_train,
#                              cv=cv, scoring=f_beta_scorer, n_jobs=-1)
#     return scores.mean()

# # 5) Run the study
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=30)

# # 6) Report best CV
# print(f"▶ Best CV F{beta:.2f}:", study.best_value)
# print("▶ Best params:", study.best_trial.params)

# # 7) Rebuild & fit final pipeline
# best = study.best_trial.params
# pipe_final = Pipeline([
#     ("pre",    preproc),
#     ("select", SelectKBest(mutual_info_classif, k=best["k"])),
#     ("clf",    xgb.XGBClassifier(
#         n_estimators=     best["n_estimators"],
#         max_depth=        best["max_depth"],
#         learning_rate=    best["learning_rate"],
#         subsample=        best["subsample"],
#         colsample_bytree= best["colsample_bytree"],
#         reg_alpha=        best["reg_alpha"],
#         reg_lambda=       best["reg_lambda"],
#         eval_metric=      "logloss",
#         random_state=     42,
#         scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
#     ))
# ])
# pipe_final.fit(X_train, y_train)

# probas_val = pipe_final.predict_proba(X_val)[:,1]
# best_thresh, best_f = 0.5, -1
# for t in np.linspace(0,1,101):
#     preds = (probas_val >= t).astype(int)
#     fβ    = fbeta_score(y_val, preds, beta=beta)
#     if fβ > best_f:
#         best_f, best_thresh = fβ, t

# print(f"Optimal threshold for F{beta:.2f}: {best_thresh:.2f}  → F{beta:.2f} = {best_f:.3f}")

# # 9) Evaluation helper
# def evaluate(split_name, X, y):
#     proba = pipe_final.predict_proba(X)[:,1]
#     preds = (proba >= best_thresh).astype(int)
#     auc   = roc_auc_score(y, proba)
#     gini  = 2*auc - 1
#     fβ    = fbeta_score(y, preds, beta=beta)
#     print(f"\n── {split_name} @ thresh={best_thresh:.2f} ──")
#     print(f"  F{beta:.2f} : {fβ:.4f}")
#     print(f"  AUC  : {auc:.4f}")
#     print(f"  Gini : {gini:.4f}")
#     print(classification_report(y, preds, digits=4))

# # 10) Run on all splits
# for name, (X, y) in [
#     ("TRAIN", (X_train, y_train)),
#     ("VAL",   (X_val,   y_val)),
#     ("TEST",  (X_test,  y_test)),
#     ("OOT1",  (X_oot1,  y_oot1)),
#     ("OOT2",  (X_oot2,  y_oot2)),
# ]:
#     evaluate(name, X, y)

# # 11) Which features survived SelectKBest?
# feat_names = pipe_final.named_steps["pre"].get_feature_names_out()
# mask       = pipe_final.named_steps["select"].get_support()
# kept       = feat_names[mask]
# dropped    = feat_names[~mask]

# print(f"\nKept {len(kept)} features:", kept.tolist())
# print(f"Dropped {len(dropped)} features:", dropped.tolist())


In [28]:
import os
import pandas as pd
import pprint

from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline          import Pipeline
import xgboost as xgb
from sklearn.metrics           import roc_auc_score, fbeta_score, precision_score, recall_score


df = pd.read_csv(
    '/app/data/final_merged.csv',
    parse_dates=['snapshot_date'],
    low_memory=False
)


train_start   = pd.Timestamp(2023,  7,  1)
train_end     = train_start + pd.DateOffset(months=11)   # → 2024-06-01
val_date      = train_end   + pd.DateOffset(months=1)    # → 2024-07-01
test_date     = val_date    + pd.DateOffset(months=1)    # → 2024-08-01
oot1_date     = test_date   + pd.DateOffset(months=1)    # → 2024-09-01
oot2_date     = oot1_date   + pd.DateOffset(months=1)    # → 2024-10-01
prod_nov_date = oot2_date   + pd.DateOffset(months=1)    # → 2024-11-01
prod_dec_date = prod_nov_date + pd.DateOffset(months=1)  # → 2024-12-01


train    = df[(df.snapshot_date >= train_start) & (df.snapshot_date <= train_end)]
val      = df[df.snapshot_date == val_date]
test     = df[df.snapshot_date == test_date]
oot1     = df[df.snapshot_date == oot1_date]
oot2     = df[df.snapshot_date == oot2_date]
prod_nov = df[df.snapshot_date == prod_nov_date]
prod_dec = df[df.snapshot_date == prod_dec_date]


drop_cols = ['Customer_ID','snapshot_date','label_def','loan_id','Name','SSN']
features  = [c for c in df.columns if c not in drop_cols + ['label']]

X_train, y_train = train[features],    train['label']
X_val,   y_val   = val[features],      val['label']
X_test,  y_test  = test[features],     test['label']
X_oot1,  y_oot1  = oot1[features],     oot1['label']
X_oot2,  y_oot2  = oot2[features],     oot2['label']
X_nov,   y_nov   = prod_nov[features], prod_nov['label']
X_dec,   y_dec   = prod_dec[features], prod_dec['label']

# 5) Separate numeric vs categorical feature lists
num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in features if pd.api.types.is_object_dtype(df[c])]


preproc = ColumnTransformer([
    ("num", StandardScaler(),              num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="drop")

best_params = {
    "n_estimators":     415,
    "max_depth":        3,
    "learning_rate":    0.009057293740640768,
    "subsample":        0.7276734855131275,
    "colsample_bytree": 0.6232189363962957,
    "reg_alpha":        0.9502195804742386,
    "reg_lambda":       1.539615751287922,
    "eval_metric":      "logloss",
    "random_state":     42,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
}

pipe_final = Pipeline([
    ("pre",    preproc),
    ("select", SelectKBest(mutual_info_classif, k=19)),
    ("clf",    xgb.XGBClassifier(**best_params))
])

pipe_final.fit(X_train, y_train)


best_thresh = 0.32
beta        = 2.0
print(f"Using fixed threshold for F{beta:.0f}: {best_thresh:.2f}")


splits = {
    "TRAIN":    (X_train, y_train),
    "VAL":      (X_val,   y_val),
    "TEST":     (X_test,  y_test),
    "OOT1":     (X_oot1,  y_oot1),
    "OOT2":     (X_oot2,  y_oot2),
    "PROD_NOV": (X_nov,   y_nov),
    "PROD_DEC": (X_dec,   y_dec),
}

results = {}
for name, (X, y) in splits.items():
    proba    = pipe_final.predict_proba(X)[:, 1]
    preds    = (proba >= best_thresh).astype(int)
    results[name] = {
        "n_rows":    len(y),
        "auc":       round(roc_auc_score(y, proba),  4),
        "gini":      round(2 * roc_auc_score(y, proba) - 1, 4),
        "f2":        round(fbeta_score(y, preds, beta=beta), 4),
        "precision": round(precision_score(y, preds), 4),
        "recall":    round(recall_score(y, preds),    4),
    }


slice_dates = {
    "train_start":   str(train_start.date()),
    "train_end":     str(train_end.date()),
    "val_date":      str(val_date.date()),
    "test_date":     str(test_date.date()),
    "oot1_date":     str(oot1_date.date()),
    "oot2_date":     str(oot2_date.date()),
    "prod_nov_date": str(prod_nov_date.date()),
    "prod_dec_date": str(prod_dec_date.date()),
}


model_artefact = {
    "model":         pipe_final,
    "model_version": "xgboostv1",
    "slice_dates":   slice_dates,
    "data_stats": {
        "X_train":         X_train.shape[0],
        "X_val":           X_val.shape[0],
        "X_test":          X_test.shape[0],
        "X_oot1":          X_oot1.shape[0],
        "X_oot2":          X_oot2.shape[0],
        "X_prod_nov":      X_nov.shape[0],
        "X_prod_dec":      X_dec.shape[0],
        "y_train_rate":    round(y_train.mean(), 2),
        "y_val_rate":      round(y_val.mean(),   2),
        "y_test_rate":     round(y_test.mean(),  2),
        "y_oot1_rate":     round(y_oot1.mean(),  2),
        "y_oot2_rate":     round(y_oot2.mean(),  2),
        "y_prod_nov_rate": round(y_nov.mean(),   2),
        "y_prod_dec_rate": round(y_dec.mean(),   2),
    },
    "results": results,
}


pprint.pprint(model_artefact)

Using fixed threshold for F2: 0.32
{'data_stats': {'X_oot1': 511,
                'X_oot2': 513,
                'X_prod_dec': 498,
                'X_prod_nov': 491,
                'X_test': 518,
                'X_train': 5958,
                'X_val': 485,
                'y_oot1_rate': np.float64(0.33),
                'y_oot2_rate': np.float64(0.28),
                'y_prod_dec_rate': np.float64(0.31),
                'y_prod_nov_rate': np.float64(0.3),
                'y_test_rate': np.float64(0.28),
                'y_train_rate': np.float64(0.28),
                'y_val_rate': np.float64(0.3)},
 'model': Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['fe_1', 'fe_2', 'fe_3',
                                                   'fe_4', 'fe_5', 'fe_6',
                                                   'fe_7', 'fe_8', 'fe_9',
                                                   'fe_

## save artefact to model bank

In [29]:
import os
import pickle

# 1) Ensure model_bank directory exists
model_bank_directory = "model_bank/"
os.makedirs(model_bank_directory, exist_ok=True)

# 2) Build the full path using your desired version name
file_path = os.path.join(model_bank_directory, "xgboostv1.pkl")

# 3) Serialize the artifact dict
with open(file_path, "wb") as f:
    pickle.dump(model_artefact, f)

print(f"Model artifact saved to {file_path}")

Model artifact saved to model_bank/xgboostv1.pkl


In [31]:
## Inference

In [30]:
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1) Load the artifact you just saved
artifact_path = "model_bank/xgboostv1.pkl"
with open(artifact_path, "rb") as f:
    model_artefact = pickle.load(f)

pipe = model_artefact["model"]
slice_dates = model_artefact["slice_dates"]

# 2) Reload the merged data (same as training)
df = pd.read_csv(
    "/app/data/final_merged.csv",
    parse_dates=["snapshot_date"],
    low_memory=False
)

# 3) Pick the OOT1 date from your saved slice_dates
oot1_date = pd.to_datetime(slice_dates["oot1_date"])

# 4) Subset to your OOT1 slice
oot1_df = df[df.snapshot_date == oot1_date]

# 5) Prepare X and y just like at train time
drop_cols = ['Customer_ID','snapshot_date','label_def','loan_id','Name','SSN']
feature_cols = [c for c in df.columns if c not in drop_cols + ['label']]

X_oot1 = oot1_df[feature_cols]
y_oot1 = oot1_df["label"]

# 6) Run inference
y_pred_proba = pipe.predict_proba(X_oot1)[:, 1]

# 7) Evaluate
oot1_auc = roc_auc_score(y_oot1, y_pred_proba)
print(f"OOT1 ({oot1_date.date()}) AUC: {oot1_auc:.4f}")

print("Inference complete — model loaded and scored successfully!")

OOT1 (2024-09-01) AUC: 0.7933
Inference complete — model loaded and scored successfully!


In [32]:
import pickle

# 1) Load the artifact
artifact_path = "model_bank/xgboostv1.pkl"
with open(artifact_path, "rb") as f:
    model_artefact = pickle.load(f)

# 2) Grab the pipeline
pipe = model_artefact["model"]

# 3) Get the post-preprocessing feature names
preprocessor = pipe.named_steps["pre"]
feat_names    = preprocessor.get_feature_names_out()

# 4) Apply the SelectKBest mask
selector      = pipe.named_steps["select"]
kept          = feat_names[selector.get_support()]

# 5) Print the 19 kept feature names
print("Selected 19 features:")
for feat in kept:
    print(" -", feat)

Selected 19 features:
 - num__Annual_Income
 - num__Num_Bank_Accounts
 - num__Num_Credit_Card
 - num__Interest_Rate
 - num__Num_of_Loan
 - num__Delay_from_due_date
 - num__Num_of_Delayed_Payment
 - num__Num_Credit_Inquiries
 - num__Outstanding_Debt
 - num__Monthly_Balance
 - num__days_overdue_per_late_payment
 - num__Credit_History_Age_num
 - num__debt_to_income_ratio
 - num__monthly_repayment_to_income
 - num__credit_inquiries_per_year
 - num__Student_Loan_count
 - cat__Credit_Mix_Bad
 - cat__Payment_of_Min_Amount_No
 - cat__Payment_of_Min_Amount_Yes
